<a href="https://colab.research.google.com/github/thproger/Th/blob/main/MathModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-huggingface sentence-transformers chromadb
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trx<0.2.0" trl peft accelerate bitsandbytes
!pip install unsloth_zoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/

In [2]:
from datasets import load_dataset

ds = load_dataset('qwedsacf/competition_math', split='train')

README.md:   0%|          | 0.00/4.82k [00:00<?, ?B/s]

data/train-00000-of-00001-7320a6f3aba8eb(…): reconstructing file:   0%|          |  0.00B / 4.85MB            

data/train-00000-of-00001-7320a6f3aba8eb(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/12500 [00:00<?, ? examples/s]

In [6]:
documents = []
metadata = []
for item in ds:
  documents.append(f"PROBLEM:\n{item['problem']}\nSOLUTION:\n{item['solution']}\n")
  metadata.append({
    'level': item['level'],
    'type': item['type']
  })

print(len(documents))
ids = [f"id_{i}" for i in range(len(documents))]

12500


In [7]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sentence_transformers import SentenceTransformer

client = chromadb.PersistentClient(path="./math_db")
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')
collection = client.get_or_create_collection(
    name='math'
)

BATCH_SIZE = 2500
total = len(documents)

for i in range(0, total, BATCH_SIZE):
    end_i = min(i + BATCH_SIZE, total)

    batch_docs = documents[i:end_i]

    batch_embeddings = model.encode(batch_docs, show_progress_bar=True, device='cuda').tolist()

    collection.add(
        documents=batch_docs,
        ids=ids[i:end_i],
        metadatas=metadata[i:end_i],
        embeddings=batch_embeddings
    )
    print(f"Load {end_i}/{total}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Load 2500/12500...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Load 5000/12500...


...

Load 7500/12500...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Load 10000/12500...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Load 12500/12500...


In [8]:
import json
import chromadb
import torch
from sentence_transformers import SentenceTransformer

# 1. Ініціалізація
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)

client = chromadb.PersistentClient(path="./math_db")
collection = client.get_or_create_collection(name='math')

# 2. Новий список кандидатів (Геометрія, Комбінаторика, Інтеграли)
fresh_candidate_problems = [
    {
        "problem": "Find the area of the region enclosed by the curves $y = x^2$ and $y = 2x - x^2$.",
        "solution": "First, find the points of intersection by setting $x^2 = 2x - x^2 \\Rightarrow 2x^2 - 2x = 0 \\Rightarrow 2x(x - 1) = 0$. The limits of integration are $x = 0$ and $x = 1$.\nThe area is given by the integral:\n$$A = \\int_{0}^{1} ((2x - x^2) - x^2) \\, dx = \\int_{0}^{1} (2x - 2x^2) \\, dx = \\left[ x^2 - \\frac{2}{3}x^3 \\right]_{0}^{1} = 1 - \\frac{2}{3} = \\frac{1}{3}.$$\nFinal Answer: $\\boxed{\\frac{1}{3}}$",
        "level": "Level 3",
        "type": "Calculus"
    },
    {
        "problem": "How many distinct anagrams can be formed using all the letters in the word 'MATHEMATICS'?",
        "solution": "The word 'MATHEMATICS' has 11 letters in total, with repeating letters: M (2), A (2), T (2).\nThe number of distinct permutations is given by the multiset coefficient:\n$$\\frac{11!}{2! \\cdot 2! \\cdot 2!} = \\frac{39,916,800}{8} = 4,989,600.$$\nFinal Answer: $\\boxed{4989600}$",
        "level": "Level 2",
        "type": "Counting & Probability"
    },
    {
        "problem": "Solve the initial value problem $\\frac{dy}{dx} = 3x^2 y$ with $y(0) = 2$.",
        "solution": "Separate the variables:\n$$\\frac{1}{y} dy = 3x^2 dx$$\nIntegrate both sides:\n$$\\ln|y| = x^3 + C \\Rightarrow y = C e^{x^3}$$\nApply the initial condition $y(0) = 2$:\n$$2 = C e^0 \\Rightarrow C = 2$$\nThus, the solution is $y(x) = 2e^{x^3}$.\nFinal Answer: $\\boxed{2e^{x^3}}$",
        "level": "Level 3",
        "type": "Calculus"
    },
    {
        "problem": "Calculate the sum of the infinite geometric series $5 + \\frac{5}{3} + \\frac{5}{9} + \\frac{5}{27} + \\dots$",
        "solution": "The first term is $a = 5$ and the common ratio is $r = \\frac{1}{3}$. Since $|r| < 1$, the sum converges:\n$$S = \\frac{a}{1 - r} = \\frac{5}{1 - 1/3} = \\frac{5}{2/3} = \\frac{15}{2}.$$\nFinal Answer: $\\boxed{\\frac{15}{2}}$",
        "level": "Level 2",
        "type": "Algebra"
    },
    {
        "problem": "Find the eigenvalues of the matrix $B = \\begin{pmatrix} 4 & 1 \\\\ 2 & 3 \\end{pmatrix}$.",
        "solution": "Solve the characteristic equation $\\det(B - \\lambda I) = 0$:\n$$\\det\\begin{pmatrix} 4 - \\lambda & 1 \\\\ 2 & 3 - \\lambda \\end{pmatrix} = (4 - \\lambda)(3 - \\lambda) - 2 = 0$$\n$$\\lambda^2 - 7\\lambda + 12 - 2 = 0 \\Rightarrow \\lambda^2 - 7\\lambda + 10 = 0$$\nFactor the quadratic:\n$$(\\lambda - 2)(\\lambda - 5) = 0$$\nSo the eigenvalues are $\\lambda_1 = 2$ and $\\lambda_2 = 5$.\nFinal Answer: $\\boxed{2, 5}$",
        "level": "Level 3",
        "type": "Linear Algebra"
    }
]

# 3. Фільтрація та збереження
DISTANCE_THRESHOLD = 0.25
existing_count = collection.count()
new_added_count = 0
sft_new_entries = []

for item in fresh_candidate_problems:
    problem = item["problem"]
    solution = item["solution"]

    query_vec = embedder.encode([problem], device=device).tolist()

    results = collection.query(
        query_embeddings=query_vec,
        n_results=2,
        include=["distances", "documents"]
    )

    min_distance = results['distances'][0][0] if results['distances'][0] else 1.0

    if min_distance > DISTANCE_THRESHOLD:
        print(f"✨ НОВА ЗАДАЧА (Min Dist: {min_distance:.3f}): {problem[:40]}...")

        new_id = f"id_fresh_{existing_count + new_added_count}"
        doc_str = f"PROBLEM:\n{problem}\n\nSOLUTION:\n{solution}"

        collection.add(
            ids=[new_id],
            documents=[doc_str],
            metadatas=[{"level": item["level"], "type": item["type"]}],
            embeddings=query_vec
        )

        retrieved_docs = results['documents'][0]
        context_str = "\n\n---\n\n".join(retrieved_docs)

        sample = {
            "instruction": "Ти — викладач вищої математики. Покроково розв'яжи задачу студента, спираючись на аналогічні приклади з контексту. Усі формули записуй у LaTeX.",
            "input": f"CONTEXT:\n{context_str}\n\nSTUDENT PROBLEM:\n{problem}",
            "output": f"РОЗВ'ЯЗОК:\n{solution}"
        }
        sft_new_entries.append(sample)
        new_added_count += 1
    else:
        print(f"⚠️ ДУБЛІКАТ/СХОЖА (Min Dist: {min_distance:.3f}): {problem[:40]}... (пропущено)")

if sft_new_entries:
    with open("rag_math_sft.jsonl", "a", encoding="utf-8") as f:
        for entry in sft_new_entries:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"\nЗавершено! Додано {new_added_count} нових унікальних задач.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✨ НОВА ЗАДАЧА (Min Dist: 0.889): Find the area of the region enclosed by ...
✨ НОВА ЗАДАЧА (Min Dist: 0.918): How many distinct anagrams can be formed...
✨ НОВА ЗАДАЧА (Min Dist: 0.865): Solve the initial value problem $\frac{d...
✨ НОВА ЗАДАЧА (Min Dist: 0.444): Calculate the sum of the infinite geomet...
✨ НОВА ЗАДАЧА (Min Dist: 1.066): Find the eigenvalues of the matrix $B = ...

Завершено! Додано 5 нових унікальних задач.


In [9]:
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Math-7B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    dtype = None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

dataset = load_dataset("json", data_files="rag_math_sft.jsonl", split="train")

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []

    for inst, inp, out in zip(instructions, inputs, outputs):
        text = (
            f"<|im_start|>system\n{inst}<|im_end|>\n"
            f"<|im_start|>user\n{inp}<|im_end|>\n"
            f"<|im_start|>assistant\n{out}<|im_end|>"
        )
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 150,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1653: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.7: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.10.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Generating train split: 0 examples [00:00, ? examples/s]

...

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5 | Num Epochs = 150 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.590208
20,0.819019
30,0.185108
40,0.037593
50,0.021330
60,0.018091
70,0.014239
80,0.007348


KeyboardInterrupt: 

In [ ]:
model.push_to_hub_gguf(
    "ftcserhiy/qwen2.5_math_rag",
    tokenizer,
    quantization_method = "q4_k_m"
)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_jdf9cqzb/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00004.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model-00002-of-00004.safetensors: downloading bytes:           |  0.00B            



Unsloth: Preparing safetensor model files:  50%|█████     | 2/4 [07:32<07:39, 229.68s/it]

model-00003-of-00004.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model-00003-of-00004.safetensors: downloading bytes:           |  0.00B            